# Bloque 1 de ITCHI v0.2.1: P95 como máximo, umbrales de R24 y A72, y corte del viento en 12 m/s

Este notebook muestra **qué cambia en cada experimento** (parte A) y **cómo se valida contra las declaratorias** (parte B). Cada figura trae una nota de cómo leerla. El diseño completo y los criterios de decisión, fijados antes de ver resultados, están en `docs/experimento_b1.md`.

**Entradas** (parte B). Son salidas de los scripts, que hay que correr antes:
- `scripts/negativos_w06.py --guardar-pasos` → `pares_municipio_ct_w06.csv.gz` y `pasos_municipio_ct_w06.csv.gz`
- `scripts/percentiles_6h.py` → `mswep_v3_{6,24,72}h_percentiles_may_nov.nc`

La parte A no necesita datos: usa las funciones de la librería con los percentiles medianos de México.

**Salidas.** Figuras en `outputs/figures/experimento_b1/` y tablas en `outputs/diagnostics/experimento_b1/`.

## Los experimentos

| | Qué cambia | Variantes | Configuración | Figuras |
|---|---|---|---|---|
| **E1. P95 como máximo** | El peligro por lluvia llega a 1 en el P95 de 6 h (v0.2 llegaba en el P99) | M0: solo P6 (P75 → 0, P90 → 0.5, P95 → 1) | — | A1, B2, B3 |
| **E2. Umbrales de R24 y A72** | La lluvia persistente (R24) y el suelo húmedo (A72) suben las clases bajas | M1: la regla de v0.2 una clase antes · rejilla de umbrales · M2: máximo de las tres duraciones · M3: tabla calibrada con declaratorias | `p1_*.yaml` (M1) | A1, A2, B1, B2, B4 |
| **E3. Corte del viento en 12 m/s** | La cola de viento termina donde V = 12 m/s, no en R_W06 (2 m/s) | W1 = 12 m/s; 17 kt de referencia | `p*_w1.yaml` | A3, B5, B6 |
| **E4. Diseño 2 × 2** | Lluvia {v0.2, M1} × viento {R_W06, 12 m/s}: si los efectos se suman o interactúan | 4 celdas | `configs/experiments/b1/` | B5, B6 |

## Cómo se valida (parte B)

- **Unidad: par municipio–ciclón.** Un par es un municipio que quedó dentro de R_W06 del ciclón en algún paso. Son 381 929 pares, 2 % con declaratoria.
- **Índice del par.** Es el máximo de ITCHI durante el evento, convertido a clase SIAT: 1 si es > 0, 2 si es > 0.2, …, 5 si es > 0.8.
- **Alerta.** Se alerta si la clase es ≥ k, y se compara con si hubo declaratoria.
- **PSS = POD − POFD.** Mide qué tan bien separa la alerta los pares con y sin declaratoria; 0 es sin habilidad. **AUC**: probabilidad de que un par con declaratoria tenga más peligro que uno sin ella.
- **ΔPSS con intervalo de 90 %.** Se calcula remuestreando ciclones, no pares, porque los pares de un mismo ciclón no son independientes. Si el intervalo no cruza 0, la diferencia es robusta.
- **ΔPOD a igual POFD.** Si es ≈ 0, el esquema nuevo **solo alerta más**: se mueve sobre la misma curva ROC y no distingue mejor.

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from itchi.config import get_project_root

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
import experimento_b1 as b1      # noqa: E402
import figuras_b1 as F           # noqa: E402

FIG = ROOT / "outputs" / "figures" / "experimento_b1"
FIG.mkdir(parents=True, exist_ok=True)
CFG_B1 = b1.ConfigB1()           # rutas, variantes de viento, bootstrap (n_boot = 1000)
RECALCULAR = False               # True: vuelve a correr el experimento aunque existan las tablas
pd.set_option("display.max_colwidth", 140)


def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()

# Parte A. Qué cambia en cada experimento (sin datos)

## A1. Peligro por lluvia de cada esquema

Cada panel es una situación de lluvia previa. La línea azul es v0.2, y las demás son los esquemas nuevos, todos con P95 como máximo.

**Cómo leerla:**
- **Panel 1, lluvia aislada.** Muestra E1: los esquemas nuevos llegan a 1 en el P95 (≈ 29 mm); v0.2 llega en el P99 (≈ 52 mm).
- **Paneles 2 a 4.** Muestran E2, cómo actúan R24 y A72:
  - M1 solo cambia si hay **persistencia y suelo húmedo a la vez** (panel 3); ahí el P90 ya es máximo.
  - M2 sube a 1 en cuanto hay lluvia ≥ P75 con R24 o A72 altos (paneles 2 y 4). Es el esquema más agresivo.
- **En ningún esquema hay peligro con P6 < P75.**

In [ ]:
guardar(F.fig_a1_lluvia_esquemas(), "A1_lluvia_esquemas")

## A2. Clase de peligro por lluvia en el plano P6 × R24

Cada fila es un esquema y cada columna un valor de A72. El color es la clase SIAT que alcanza el peligro por lluvia. La zona gris es imposible, porque R24 incluye a P6.

**Cómo leerla.** La frontera roja (clase 5) se mueve hacia la izquierda conforme el esquema permite que R24 y A72 escalen:
- **v0.2:** solo se mueve con P6 ≥ P95.
- **M1:** se mueve desde P75 si R24 ≥ P95 y hay antecedente.
- **M2:** casi todo es clase 5 con A72 ≥ P95.

Si la parte B muestra que M2 tiene demasiadas falsas alarmas, esta figura explica por qué.

In [ ]:
guardar(F.fig_a2_tablas(), "A2_tablas_clase")

## A3. Dónde termina la cola de viento (E3)

H_W contra la distancia al centro, para un huracán y para una tormenta tropical. Dentro de R34 las tres variantes son iguales. Afuera, la cola llega a cero en R_W06 (W0), en 12 m/s (W1) o en 17 kt.

**Cómo leerla.** Con W0, casi todo el radio de ~800 km tiene peligro por viento > 0. Eso es lo que saturó la clase 1 en v0.2: el 96 % de los pares tenía viento. Con 12 m/s, la cola termina unos 100 km fuera de R34.

In [ ]:
guardar(F.fig_a3_viento(), "A3_cola_viento")

# Parte B. Validación contra declaratorias

Esta celda corre el experimento (unos minutos con los pasos) o lee las tablas si ya existen. El modo **pasos** evalúa el máximo del índice durante el evento. El modo **pico** solo evalúa el pico de P6, y subestima a M1 y M2.

In [ ]:
out = Path(CFG_B1.out_dir)
archivos = ["celdas_P6_R24_A72.csv", "lluvia_candidatos.csv", "diseno_2x2.csv", "por_par.csv.gz"]
if RECALCULAR or not all((out / f).exists() for f in archivos):
    res = b1.run(CFG_B1)
    celdas, lluvia, diseno, por_par = res["celdas"], res["lluvia"], res["diseno"], res["por_par"]
else:
    celdas = pd.read_csv(out / "celdas_P6_R24_A72.csv", index_col=[0, 1, 2])
    lluvia = pd.read_csv(out / "lluvia_candidatos.csv")
    diseno = pd.read_csv(out / "diseno_2x2.csv")
    por_par = pd.read_csv(out / "por_par.csv.gz", dtype={"CVEGEO": str})
print(f"pares: {len(por_par):,} | con declaratoria: {int(por_par['D'].sum()):,} ({100 * por_par['D'].mean():.1f} %)")
print((out / "resumen_b1.md").read_text(encoding="utf-8").splitlines()[2])

## B1. Evidencia: ¿cuánto pesan los umbrales de R24 y A72? (E2)

Probabilidad de declaratoria (y número de pares) en cada combinación de clases. Cada variable se compara con sus propios percentiles. Es el dato crudo, antes de cualquier esquema.

**Cómo leerla:**
- **Dentro de un panel:** si el color sube de izquierda a derecha en una fila, R24 aporta. Si sube de un panel al siguiente en la misma celda, A72 aporta.
- **Fila decisiva:** importa sobre todo P90–P95, porque P95 ya es el máximo en todos los esquemas.
- **Celdas sin color:** tienen menos de 30 pares y no se interpretan.

In [ ]:
guardar(F.fig_b1_celdas(celdas), "B1_celdas_P6_R24_A72")

## B2. Comparación de los esquemas de lluvia (E1 y E2)

- **Panel 1:** ΔPSS contra v0.2 para alerta en clase ≥ 1, ≥ 3 y ≥ 5. Responde E1.
- **Panel 2:** ΔPSS contra M0, que es el aporte de R24 y A72 sobre la lluvia de 6 h sola. Responde E2.
- **Panel 3:** el costo, es decir, cuántas alertas de clase 5 se emiten por cada declaratoria (sesgo) y qué fracción de pares sin declaratoria recibe clase 5 (POFD).

**Cómo leerla.** Una barra cuyo intervalo no cruza 0 es una diferencia robusta. Un esquema que gana PSS pero triplica el sesgo gana porque alerta más; la figura B3 dice si además distingue mejor.

In [ ]:
guardar(F.fig_b2_candidatos(lluvia), "B2_candidatos_lluvia")

## B3. Curvas ROC: ¿mejor discriminación o solo más alertas?

Cada curva es un esquema de lluvia; los puntos numerados son las clases 1 a 5.

**Cómo leerla:**
- **Curva por encima de la de v0.2:** el esquema distingue mejor.
- **Misma curva, puntos desplazados hacia la derecha:** el esquema solo alerta más.
- **Puntos sobre la curva de v0.2:** si el punto 5 de M0 cae sobre la curva de v0.2, P95 como máximo es una decisión operativa (más alertas por el subregistro), no una mejora de habilidad.

In [ ]:
guardar(F.fig_b3_roc(por_par), "B3_roc_lluvia")

## B4. Impacto de cada umbral de R24 y A72 (rejilla de M1)

La regla de M1 es «P90 ya es máximo si R24 ≥ X y A72 ≥ Y». Cada celda es una combinación (X, Y), con su ΔPSS contra M0 y el intervalo de 90 %. A la derecha se quita una condición a la vez (ablación).

**Cómo leerla.** Si «solo R24» rinde igual que M1, el antecedente A72 no aporta, y viceversa. La rejilla es **sensibilidad**: no se elige el mejor punto con los mismos pares, porque eso sobreajusta.

In [ ]:
guardar(F.fig_b4_rejilla(lluvia), "B4_rejilla_umbrales_M1")

## B5. Diseño 2 × 2: lluvia × viento (E3 y E4)

PSS para alerta en clase ≥ 1, ≥ 3 y ≥ 5, y AUC, en cada combinación. Las barras son el intervalo de 90 % de la diferencia contra v0.2 con cola hasta R_W06. Se agrega 17 kt como referencia.

**Cómo leerla:**
- **E3:** el corte en 12 m/s debería recuperar el PSS de la clase 1, que con W0 es ≈ 0.
- **AUC:** no debe caer por debajo de la línea punteada (v0.2 sin viento); si cae, el viento dejó de aportar información.
- **E4:** si las líneas de v0.2 y M1 son paralelas, los dos cambios no interactúan y se pueden decidir por separado.

In [ ]:
guardar(F.fig_b5_2x2(diseno), "B5_diseno_2x2")

## B6. Cuántas alertas emite cada configuración

Porcentaje de pares en cada clase máxima de ITCHI. A la izquierda los pares sin declaratoria y a la derecha los pares con declaratoria.

**Cómo leerla.** Una buena configuración pone más rojo a la derecha que a la izquierda. Si un cambio sube el rojo en ambos lados por igual, solo alerta más.

In [ ]:
guardar(F.fig_b6_frecuencias(por_par), "B6_frecuencias_clase")

## B7. Decisiones según los criterios fijados antes de ver resultados

Esta tabla aplica los criterios de `docs/experimento_b1.md` (sección 6) a los resultados. Es el resumen para presentar.

In [ ]:
dec = F.decisiones(lluvia, diseno)
dec.to_csv(FIG / "B7_decisiones.csv", index=False)
dec

## B8. Tablas completas

In [ ]:
cols = ["lluvia", "AUC_anual", "PSS>=1", "PSS>=3", "PSS>=5", "POD>=5", "POFD>=5", "sesgo>=5", "SEDI>=5",
        "dPSS>=5", "dPSS>=5 IC90", "dPOD@POFD>=5", "dPSS>=5 vs M0", "dPSS>=5 vs M0 IC90"]
lluvia[cols].round(3)

In [ ]:
cols = ["lluvia", "viento", "AUC_anual", "% pares con H_W > 0", "PSS>=1", "PSS>=3", "PSS>=5",
        "dPSS>=1", "dPSS>=1 IC90", "dPSS>=5", "dPSS>=5 IC90", "sesgo>=5"]
diseno[cols].round(3)

# Parte C. Siguiente paso: casos de estudio en malla

Las configuraciones del 2 × 2 sirven igual en malla. Para Cristóbal 2020 y Odile 2014 (y después Emily, Alex y Patricia), en los notebooks 07 y 08 se cambia el cálculo por:

```python
from itchi.config import itchi_v02_settings_from_config, load_experiment_config
from itchi.pipeline import compute_itchi_snapshot_v02

settings = itchi_v02_settings_from_config(load_experiment_config(ROOT / "configs/experiments/b1/p1_w1.yaml"))
res = compute_itchi_snapshot_v02(p6, r24, a72, quantiles_6h=q6, quantiles_24h=q24, quantiles_72h=q72,
                                 lon=lon2d, lat=lat2d, center_lon=row.lon, center_lat=row.lat,
                                 vmax_kt=row.vmax_kt, rmw_km=row.rmw_km,
                                 r34_by_quadrant=r34_nm, rocloud_by_quadrant=rocloud_km, **settings)
```

Ahí el viento sí usa R34 por cuadrante y el perfil calibrado, que en la parte B se aproximan.